In [1]:
def generate_candidates_and_rerank(prompt, tokenizer, policy_model, reward_model, num_candidates=5):
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(policy_model.device)

    candidates = []
    for _ in range(num_candidates):
        outputs = policy_model.generate(input_ids, max_new_tokens=50, do_sample=True, top_k=50, top_p=0.95)
        decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
        candidates.append(decoded)

    # Score candidates with reward model
    scored = []
    for response in candidates:
        # Reward model input = prompt + response
        inputs = tokenizer(prompt + response, return_tensors="pt", truncation=True, padding=True).to(reward_model.device)
        with torch.no_grad():
            reward = reward_model(**inputs).item()
        scored.append((response, reward))

    # Pick highest reward
    best_response = max(scored, key=lambda x: x[1])[0]
    return best_response


In [9]:
import torch.nn as nn
from transformers import AutoModel

class RewardModel(nn.Module):
    def __init__(self, base_model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0"):
        super().__init__()
        self.base_model = AutoModel.from_pretrained(base_model_name)
        self.reward_head = nn.Linear(self.base_model.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state  # [batch, seq_len, hidden]
        last_hidden = hidden_states[:, -1, :]      # [batch, hidden]
        reward = self.reward_head(last_hidden)     # [batch, 1]
        return reward.squeeze(-1)


In [2]:
import torch
import torch.nn as nn
from transformers import AutoModel

class RewardModel(nn.Module):
    def __init__(self, base_model_name="TinyLlama/TinyLlama-1.1B-Chat"):
        super().__init__()
        self.base_model = AutoModel.from_pretrained(base_model_name)
        self.reward_head = nn.Linear(self.base_model.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state  # (batch, seq_len, hidden)
        last_hidden = hidden_states[:, -1, :]      # take last token hidden state
        reward = self.reward_head(last_hidden)     # (batch, 1)
        return reward.squeeze(-1)


C:\Users\gadep\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "TheBloke/TinyLlama-1.1B-intermediate-step-715k-1.5T-GPTQ"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="auto", device_map="auto")


C:\Users\gadep\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
CUDA extension not installed.
CUDA extension not installed.
C:\Users\gadep\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\gadep\.cache\huggingface\hub\models--TheBloke--TinyLlama-1.1B-intermediate-step-715k-1.5T-GPTQ. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLI

In [3]:
import torch
from transformers import AutoTokenizer
from auto_gptq import AutoGPTQForCausalLM

model_id = "TheBloke/TinyLlama-1.1B-intermediate-step-715k-1.5T-GPTQ"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoGPTQForCausalLM.from_quantized(
    model_id,
    device="cuda" if torch.cuda.is_available() else "cpu",
    use_safetensors=True,
    trust_remote_code=True,
)


WARNING - Exllamav2 kernel is not installed, reset disable_exllamav2 to True. This may because you installed auto_gptq using a pre-build wheel on Windows, in which exllama_kernels are not compiled. To use exllama_kernels to further speedup inference, you can re-install auto_gptq from source.
WARNING - CUDA kernels for auto_gptq are not installed, this will result in very slow inference speed. This may because:
1. You disabled CUDA extensions compilation by setting BUILD_CUDA_EXT=0 when install auto_gptq from source.
2. You are using pytorch without CUDA support.
3. CUDA and nvcc are not installed in your device.
C:\Users\gadep\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn

In [4]:
import torch
import torch.nn as nn
from transformers import AutoModel

class RewardModel(nn.Module):
    def __init__(self, base_model_name="TheBloke/TinyLlama-1.1B-intermediate-step-715k-1.5T-GPTQ"):
        super().__init__()
        self.base_model = AutoModel.from_pretrained(base_model_name, trust_remote_code=True)
        self.reward_head = nn.Linear(self.base_model.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state[:, -1, :]  # [batch, hidden]
        reward = self.reward_head(last_hidden)             # [batch, 1]
        return reward.squeeze(-1)


In [7]:
import torch
import torch.nn as nn
from transformers import AutoModel

class RewardModel(nn.Module):
    def __init__(self, base_model_name="TheBloke/TinyLlama-1.1B-intermediate-step-715k-1.5T-GPTQ"):
        super().__init__()
        self.base_model = AutoModel.from_pretrained(base_model_name, trust_remote_code=True)
        self.reward_head = nn.Linear(self.base_model.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state[:, -1, :]  # [batch, hidden]
        reward = self.reward_head(last_hidden)             # [batch, 1]
        return reward.squeeze(-1)

# Save the model
reward_model = RewardModel()
torch.save(reward_model.state_dict(), "reward_model.pth")
print(" Reward model saved as 'reward_model.pth'")


Found modules on cpu/disk. Using Exllama/Exllamav2 backend requires all the modules to be on GPU. Setting `disable_exllama=True`


 Reward model saved as 'reward_model.pth'


In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Re-initialize architecture
loaded_reward_model = RewardModel()
loaded_reward_model.load_state_dict(torch.load("reward_model.pth", map_location=device))
loaded_reward_model.to(device)
loaded_reward_model.eval()

print("Reward model loaded and ready for inference.")


Found modules on cpu/disk. Using Exllama/Exllamav2 backend requires all the modules to be on GPU. Setting `disable_exllama=True`


Reward model loaded and ready for inference.


In [10]:
import torch
from transformers import AutoTokenizer

# Example model ID (replace with your actual model)
model_id = "TheBloke/TinyLlama-1.1B-intermediate-step-715k-1.5T-GPTQ"

# Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# If the tokenizer has no pad_token, set pad_token = eos_token (common for causal LMs)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Example texts to tokenize
texts = ["Why is the sky blue?", "Tell me a joke."]

# Tokenize with padding and truncation enabled
encodings = tokenizer(
    texts,
    return_tensors="pt",
    padding=True,      # pad to longest in batch
    truncation=True,   # truncate if too long
).to(device)

# At this point, `encodings` is ready to be fed to your model
# For example, if you had a reward model:
# reward_scores = reward_model(**encodings)

print("Input IDs shape:", encodings['input_ids'].shape)
print("Attention mask shape:", encodings['attention_mask'].shape)

# Your inference code here...


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Input IDs shape: torch.Size([2, 7])
Attention mask shape: torch.Size([2, 7])


In [14]:
def compute_reward(texts, tokenizer, reward_model, device="cpu"):
    # Set pad token if not set
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Tokenize and move to device
    encodings = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
    )
    encodings = {k: v.to(device) for k, v in encodings.items()}

    # Cast inputs to model's dtype
    dtype = next(reward_model.parameters()).dtype
    encodings = {k: v.to(dtype) for k, v in encodings.items()}

    # Forward pass with no_grad
    with torch.no_grad():
        outputs = reward_model(
            input_ids=encodings["input_ids"],
            attention_mask=encodings["attention_mask"],
        )

    # Handle output shape
    if isinstance(outputs, torch.Tensor):
        rewards = outputs.squeeze(-1)
    elif hasattr(outputs, "logits"):
        rewards = outputs.logits.squeeze(-1)
    else:
        raise ValueError("Unexpected reward_model output format")

    return rewards.cpu()


In [16]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel

class RewardModel(nn.Module):
    def __init__(self, base_model_name="TheBloke/TinyLlama-1.1B-intermediate-step-715k-1.5T-GPTQ"):
        super().__init__()
        self.base_model = AutoModel.from_pretrained(base_model_name, trust_remote_code=True)
        self.reward_head = nn.Linear(self.base_model.config.hidden_size, 1)
        # Convert reward_head to same dtype as base_model parameters to avoid dtype mismatch
        self.reward_head = self.reward_head.to(next(self.base_model.parameters()).dtype)

    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state[:, -1, :]  # [batch, hidden]
        reward = self.reward_head(last_hidden)             # [batch, 1]
        return reward.squeeze(-1)

def compute_reward(texts, tokenizer, reward_model, device="cpu"):
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    encodings = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
    )
    encodings = {k: v.to(device) for k, v in encodings.items()}

    # Only convert floating tensors (if any), keep input_ids and attention_mask as integers
    dtype = next(reward_model.parameters()).dtype
    encodings = {k: v.to(dtype) if v.dtype.is_floating_point else v for k, v in encodings.items()}

    reward_model.eval()
    with torch.no_grad():
        rewards = reward_model(
            input_ids=encodings["input_ids"],
            attention_mask=encodings["attention_mask"]
        )
    return rewards.cpu()

def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    base_model_name = "TheBloke/TinyLlama-1.1B-intermediate-step-715k-1.5T-GPTQ"

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)

    # Initialize reward model
    reward_model = RewardModel(base_model_name=base_model_name)

    # Load saved reward model weights
    reward_model.load_state_dict(torch.load("reward_model.pth", map_location=device))
    reward_model.to(device)

    # Sample texts to score
    texts = [
        "Why is the sky blue?",
        "Tell me a joke."
    ]

    rewards = compute_reward(texts, tokenizer, reward_model, device=device)
    print("Reward scores:", rewards)

if __name__ == "__main__":
    main()


Found modules on cpu/disk. Using Exllama/Exllamav2 backend requires all the modules to be on GPU. Setting `disable_exllama=True`


Reward scores: tensor([0.6050, 0.6982], dtype=torch.float16)
